***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import urllib.request, json
pd.set_option('display.max_columns', None)


import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'EPA')

path_code    = os.path.join(path_git, 'Data', 'EPA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)


***

Health_3

***

In [ ]:
indicator_name = 'Health_3'

file_name = f"{indicator_name} MSA EPA.xlsx"
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MSA')

display(df_msa.head())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

# Set Indicator
indicator_name = 'Health_3'
plot_name = 'aqi'
export = False


## Organizing ---


df_plot = df_msa.copy()

df_plot['date_local'] = pd.to_datetime(df_plot['date_local'])
df_plot['Year' ] = df_plot['date_local'].dt.year
df_plot = df_plot[df_plot['Year'] >= 1999]

df_plot = df_plot[df_plot['AQI Daily Maximum'] > 100]


df_plot = pd.DataFrame(df_plot[['MSA', 'Year']].value_counts())
df_plot = df_plot.reset_index()


df_plot = df_plot.sort_values(['MSA', 'Year'], ascending = [True, True])
df_plot = df_plot[df_plot['MSA'] == 'Sacramento--Roseville--Arden-Arcade, CA']
df_plot['moving_avg'] = df_plot['count'].rolling(window=5).mean()
df_plot = df_plot.reset_index(drop=True)


display(df_plot.head())


## Plotting ---


fig = px.bar(df_plot, x='Year', y='count')
fig['data'][0]['marker']['color']='#1F45FC'

fig.add_trace(go.Scatter(x=df_plot["Year"], y=df_plot['moving_avg']
                         , name = '5-Year Average'
                         , mode = 'lines+markers'
                         , line=go.scatter.Line(color="maroon")
                        ))


title = '<b>Days Violating National Ambient Air Quality Standards (Ozone and/or PM2.5)</b>  <br><sup>4-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range=[0, 161])
fig.update_xaxes(tick0=0, dtick=4, range=[1998.5, 2023.5])
fig.update_traces(hovertemplate='Number of days: %{y}<br>%{x}')
# fig.update_layout(showlegend=False)


plot_agol(export=export)